# Capítulo 25 · Laboratório 25: estratégias clássicas
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Regras públicas, na versão mais simples e só de compra, sem otimização de parâmetros; custo de 0,10% ida e volta. O Setup MK é uma versão pública e não mede o indicador de Márcio Kieling; o HiLo com confirmação não mede o Gibex de Giba Coelho. O CDI vem da API do Banco Central (série 4391). Na primeira execução, o notebook baixa os dados do Yahoo Finance e do Banco Central na sua máquina e guarda uma cópia local (`dados_cap25.pkl`), que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O Yahoo só guarda 730 dias de barras de 60 minutos, contados de hoje: quando a janela do livro (07/11/2024 a 30/09/2026) não cabe mais, a Parte 8 usa outra, e o notebook avisa.

In [ ]:
!pip install -q yfinance

In [ ]:
import io
import logging
import os
import urllib.request
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap25.pkl"
CUSTO = 0.001                           # 0,10% ida e volta


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


pc = lambda x, k=0: "" if pd.isna(x) else f"{x:.{k}%}".replace(".", ",")
nn = lambda x: f"{x:,}".replace(",", ".")


def ps(x, k=2):
    """Com sinal; o que arredonda para zero sai como 0,00%, sem sinal."""
    if pd.isna(x):
        return ""
    if round(x * 100, k) == 0:
        return pc(0.0, k)
    return ("+" if x > 0 else "") + pc(x, k)


B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
INTRA = ["SPY", "QQQ", "IWM", "DIA", "BOVA11.SA", "PETR4.SA", "VALE3.SA", "ITUB4.SA"]

## Parte 1: dados

In [ ]:
if os.path.exists(CACHE):
    D, IDX, CDI, H60 = pd.read_pickle(CACHE)
else:
    hoje = pd.Timestamp.today().normalize()
    D = yf.download(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    IDX = yf.download(["^BVSP", "^GSPC"], start="1999-01-01", end=FIM, auto_adjust=True, progress=False,
                      group_by="ticker")
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados?formato=csv&dataInicial=01/01/2000"
           "&dataFinal=30/09/2026")                                # CDI mensal (% no mês), Banco Central
    txt = urllib.request.urlopen(url, timeout=60).read().decode("latin-1")
    CDI = pd.read_csv(io.StringIO(txt), sep=";", decimal=",")
    # o Yahoo só guarda ~730 dias de barras de 60 minutos, contados de hoje
    ini60 = max((hoje - pd.Timedelta(days=720)).strftime("%Y-%m-%d"), "2024-11-07")   # o livro usa 07/11/2024 a 30/09/2026
    if ini60 < FIM:
        H60 = yf.download(INTRA, start=ini60, end=FIM, interval="60m", auto_adjust=True, progress=False, group_by="ticker")
    else:                               # FIM já saiu da janela: usa os últimos 720 dias até hoje
        H60 = yf.download(INTRA, start=ini60, interval="60m", auto_adjust=True, progress=False, group_by="ticker")
    confere({"as 40 ações": D, "o Ibovespa e o S&P 500": IDX, "as barras de 60 minutos": H60})
    if ini60 > "2024-11-07" or ini60 >= FIM:
        print(f"Aviso: a janela do livro é de 07/11/2024 a 30/09/2026; o Yahoo só guarda 730 dias de barras de 60 minutos;"
              f" a Parte 8 usa de {H60.index[0]:%d/%m/%Y} a {H60.index[-1]:%d/%m/%Y}, e os números diferem dos do livro.")
    pd.to_pickle((D, IDX, CDI, H60), CACHE)


def limpa(d):
    """Sem filtro de volume: o Yahoo informa volume zero em algumas barras intradiárias da B3 (a das 10h)."""
    d = d.drop(columns=["Adj Close"], errors="ignore").dropna()
    return d[(d["Close"] > 0) & (d["High"] > d["Low"])]

## Parte 2: as regras

In [ ]:
def rsi(c, n):
    """IFR de Wilder."""
    dl = c.diff()
    g = dl.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    p = (-dl.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + g / p)


def saida_por_regra(C, i, sair, prazo=10):
    """Entra no fechamento da barra i; sai no primeiro fechamento em que sair[k] for verdadeiro (ou no prazo)."""
    for k in range(i + 1, min(i + prazo + 1, len(C))):
        if sair[k]:
            return C[k] / C[i] - 1, k - i
    k = min(i + prazo, len(C) - 1)
    return C[k] / C[i] - 1, k - i


def setup_mk(O, H, L, C, m9, m20, espera=5, prazo=250):
    """Setup MK em versão pública. As médias de 9 e 20 cruzam para cima; nos 5 candles seguintes, o primeiro candle
    de alta que fecha acima das duas médias é o gatilho; compra no rompimento da máxima dele (na abertura, se abrir
    acima); stop na média de 20 do dia do gatilho; sai quando as médias cruzam para baixo (ou no fim do prazo).
    A regra de coloração do autor do setup não é pública e não está aqui. Devolve (retorno, barras, índice)."""
    res, n, i = [], len(C), 21
    while i < n - 2:
        if m9[i] > m20[i] and m9[i - 1] <= m20[i - 1]:
            for g in range(i, min(i + espera, n - 2)):
                if C[g] > m9[g] and C[g] > m20[g] and C[g] > O[g]:
                    if H[g + 1] > H[g]:
                        entrada, stop, ini = max(H[g], O[g + 1]), m20[g], g + 1
                        if stop >= entrada:
                            break
                        fim = min(ini + prazo, n) - 1
                        for k in range(ini, fim + 1):
                            if L[k] <= stop:
                                res.append((min(stop, O[k]) / entrada - 1, k - ini + 1, ini)); i = k; break
                            if m9[k] < m20[k] or k == fim:
                                res.append((C[k] / entrada - 1, k - ini + 1, ini)); i = k; break
                    break
        i += 1
    return res


def opera(t, d):
    """Todas as operações das regras do capítulo numa ação (diário)."""
    o, h, l, c = d["Open"], d["High"], d["Low"], d["Close"]
    O, H, L, C = o.to_numpy(), h.to_numpy(), l.to_numpy(), c.to_numpy()
    mm200, mm5, mm20 = c.rolling(200).mean(), c.rolling(5).mean(), c.rolling(20).mean()
    acima200 = (c > mm200).to_numpy()
    r2 = rsi(c, 2).to_numpy()
    sai_mm5 = (c > mm5).to_numpy()
    min7, max7 = (c <= c.rolling(7).min()).to_numpy(), (c >= c.rolling(7).max()).to_numpy()
    dp = c.rolling(20).std(ddof=0)                  # desvio da população, como nas bandas de Bollinger (Cap. 26)
    pctb = ((c - (mm20 - 2 * dp)) / (4 * dp)).to_numpy()
    sai_mm20 = (c > mm20).to_numpy()
    amp = h - l
    nr7 = (amp <= amp.rolling(7).min()).to_numpy()
    ema9 = c.ewm(span=9, adjust=False).mean().to_numpy()
    n, out = len(C), []
    for i in range(201, n - 11):
        ano = d.index[i].year
        # IFR(2) de Connors: acima da MM200, IFR(2) < 10; compra no fechamento; sai ao fechar acima da MM5
        if acima200[i] and r2[i] < 10:
            r, dd = saida_por_regra(C, i, sai_mm5)
            out.append((t, ano, "IFR(2) de Connors", r, dd))
        # Double Seven: acima da MM200 e fechamento na mínima de 7 dias; sai no fechamento na máxima de 7 dias
        if acima200[i] and min7[i]:
            r, dd = saida_por_regra(C, i, max7)
            out.append((t, ano, "Double Seven de Connors", r, dd))
        # %b de Bollinger abaixo de 0 em tendência de alta; sai ao fechar acima da MM20
        if acima200[i] and pctb[i] < 0:
            r, dd = saida_por_regra(C, i, sai_mm20)
            out.append((t, ano, "%b de Bollinger < 0", r, dd))
        # NR7 de Crabel: barra mais estreita de 7; amanhã compra acima da máxima, sai no fechamento
        if nr7[i] and H[i + 1] > H[i]:
            out.append((t, ano, "NR7 de Crabel", C[i + 1] / max(H[i], O[i + 1]) - 1, 1))
        # Oops! de Larry Williams: abre abaixo da mínima de ontem e volta a ela; sai no fechamento
        if O[i + 1] < L[i] and H[i + 1] >= L[i]:
            out.append((t, ano, "Oops! de Larry Williams", C[i + 1] / L[i] - 1, 1))
        # Setup 9.1: MME9 vira para cima; compra acima da máxima do sinal; stop na mínima dele;
        # sai quando a MME9 vira para baixo (ou depois de 250 pregões)
        if ema9[i] > ema9[i - 1] and ema9[i - 1] <= ema9[i - 2] and H[i + 1] > H[i]:
            entrada, stop = max(H[i], O[i + 1]), L[i]
            fim = min(i + 250, n - 1)
            for k in range(i + 1, fim + 1):
                if L[k] <= stop:
                    r, dd = min(stop, O[k]) / entrada - 1, k - i
                    break
                if (k > i + 1 and ema9[k] < ema9[k - 1]) or k == fim:
                    r, dd = C[k] / entrada - 1, k - i
                    break
            out.append((t, ano, "Setup 9.1", r, dd))
    mk = setup_mk(O, H, L, C, c.ewm(span=9, adjust=False).mean().to_numpy(), c.ewm(span=20, adjust=False).mean().to_numpy())
    out += [(t, d.index[i0].year, "Setup MK (versão pública)", r, dd) for r, dd, i0 in mk]
    # HiLo de 8 com confirmação: vira comprado quando fecha acima da média das máximas de 8 e o dia seguinte rompe
    # a máxima do dia da virada; sai quando fecha abaixo da média das mínimas de 8 e o dia seguinte rompe a mínima
    hi8, lo8 = h.rolling(8).mean().shift(1).to_numpy(), l.rolling(8).mean().shift(1).to_numpy()
    pos, entrada, ini = 0, 0.0, 0
    for i in range(210, n - 1):
        if pos == 0 and C[i] > hi8[i] and C[i - 1] <= hi8[i - 1] and H[i + 1] > H[i]:
            pos, entrada, ini = 1, max(H[i], O[i + 1]), i + 1
        elif pos == 1 and i > ini and C[i] < lo8[i] and L[i + 1] < L[i]:
            out.append((t, d.index[ini].year, "HiLo de 8 com confirmação", min(L[i], O[i + 1]) / entrada - 1, i + 1 - ini))
            pos = 0
    if pos == 1:                                                  # operação aberta no fim: sai no último fechamento
        out.append((t, d.index[ini].year, "HiLo de 8 com confirmação", C[-1] / entrada - 1, n - 1 - ini))
    return out


ORDEM = ["IFR(2) de Connors", "Double Seven de Connors", "%b de Bollinger < 0", "HiLo de 8 com confirmação",
         "Setup MK (versão pública)", "Setup 9.1", "NR7 de Crabel", "Oops! de Larry Williams"]

## Parte 3: a tabela geral

In [ ]:
SERIES = {}
for t in TICKERS:
    x = limpa(D[t])
    if len(x) > 500:
        SERIES[t] = x
ops, refs = [], []
for t, d in SERIES.items():
    ops += opera(t, d)
    refs.append(d["Close"].pct_change().mean())
R = pd.DataFrame(ops, columns=["ticker", "ano", "estrategia", "r", "dias"])
REF = float(np.mean(refs))
print(f"Parte 3. Um dia qualquer rende em média {ps(REF, 3)} nestas 40 ações (2005 a 2026)")
print("estratégia                      operações   bruto    com custo   acerto   dias   segurar pelos mesmos dias")
TAB = {}
for k in ORDEM:
    x = R[R["estrategia"] == k]
    TAB[k] = (len(x), x["r"].mean(), x["r"].mean() - CUSTO, (x["r"] > 0).mean(), x["dias"].mean(), REF * x["dias"].mean())
    print(f"{k:<30} {nn(len(x)):>9} {ps(x['r'].mean()):>8} {ps(x['r'].mean() - CUSTO):>10} {pc((x['r'] > 0).mean()):>7}"
          f" {str(round(x['dias'].mean(), 1)).replace('.', ','):>6} {ps(REF * x['dias'].mean()):>12}")

In [ ]:
import matplotlib.pyplot as plt
g = pd.DataFrame({k: {"com custo": TAB[k][2], "segurar pelos mesmos dias": TAB[k][5]} for k in ORDEM}).T
ax = g.plot(kind="barh", figsize=(9, 4), color=["black", "white"], edgecolor="black",
            title="Estratégias x comprar e segurar pelos mesmos dias (média por operação)")
ax.axvline(0, color="gray")
ax.invert_yaxis()
plt.show()

## Parte 4: por mercado e por época

In [ ]:
print("\nParte 4. Média por operação, com custo (número de operações)")
print("estratégia                      B3 2005-14       B3 2015-26       EUA 2005-14      EUA 2015-26")
R["br"] = R["ticker"].str.endswith(".SA")
for k in ORDEM:
    x = R[R["estrategia"] == k]
    cel = []
    for br in (True, False):
        for a, b in ((2005, 2014), (2015, 2026)):
            y = x[(x["br"] == br) & x["ano"].between(a, b)]["r"]
            cel.append(f"{ps(y.mean() - CUSTO)} ({nn(len(y))})")
    print(f"{k:<30} " + "  ".join(f"{c_:<15}" for c_ in cel))

## Parte 5: o diamante

In [ ]:
fases = []
for t, d in SERIES.items():
    c = d["Close"]
    mm50, mm200 = c.rolling(50).mean(), c.rolling(200).mean()
    fase = np.select([(c > mm50) & (mm50 > mm200), (c < mm50) & (mm50 > mm200) & (c > mm200),
                      (c < mm50) & (c < mm200) & (mm50 > mm200), (c < mm50) & (mm50 < mm200),
                      (c > mm50) & (mm50 < mm200) & (c < mm200), (c > mm50) & (c > mm200) & (mm50 < mm200)],
                     ["3 altista", "4 aviso", "5 distribuição", "6 baixista", "1 recuperação", "2 acumulação"], "")
    fases.append(pd.DataFrame({"fase": fase, "f20": c.shift(-20) / c - 1})[mm200.notna().to_numpy()])
F = pd.concat(fases)
F = F[F["fase"] != ""].dropna()
print("\nParte 5. Diamante (fases de Chuck Dukas, médias de 50 e 200): retorno nos 20 pregões seguintes")
for k, x in F.groupby("fase"):
    print(f"   fase {k:<16} {pc(len(x) / len(F)):>4} dos dias | média {ps(x['f20'].mean())} | positivo em {pc((x['f20'] > 0).mean())}")

## Parte 6: pivôs clássicos

In [ ]:
# "respeitou" = a mínima chega a até 0,1 ATR do S1 e o dia fecha acima dele; o nível falso usa a mesma distância
# relativa entre o S1 e o fechamento, mas tirada de outro dia (6 pregões antes)
pivos = []
for t, d in SERIES.items():
    h, l, c = d["High"], d["Low"], d["Close"]
    piv = (h + l + c) / 3
    s1 = (2 * piv - h).shift(1)
    s1_falso = c.shift(1) * ((2 * piv - h) / c).shift(6)
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    a14 = tr.rolling(14).mean().shift(1)
    for nome, nivel in [("S1 de ontem", s1), ("S1 falso (mesma distância, outro dia)", s1_falso)]:
        toca = (l <= nivel + 0.1 * a14) & (h >= nivel - 0.1 * a14)
        pivos.append((nome, int(toca.sum()), int((toca & (c > nivel)).sum())))
P = pd.DataFrame(pivos, columns=["nivel", "toca", "segura"]).groupby("nivel").sum()
print("\nParte 6. Pivôs clássicos: o dia toca o nível e fecha acima dele")
for k, x in P.iterrows():
    print(f"   {k:<38} tocou em {nn(int(x['toca'])):>7} dias | fechou acima em {pc(x['segura'] / x['toca'])}")

## Parte 7: Faber (2007)

In [ ]:
cdi = CDI.copy()
cdi.index = pd.to_datetime(cdi["data"], dayfirst=True) + pd.offsets.MonthEnd(0)
cdi = cdi["valor"].astype(float) / 100
print("\nParte 7. Faber, de 2000 a setembro de 2026: comprado quando o mês fecha acima da média de 10 meses;"
      " fora, CDI (Brasil) ou zero (EUA)")
FABER = {}
for nome, tk, caixa in [("Ibovespa", "^BVSP", cdi), ("S&P 500", "^GSPC", None)]:
    m = limpa(IDX[tk])["Close"].resample("ME").last()
    sinal = (m > m.rolling(10).mean()).shift(1, fill_value=False).astype(bool)
    ret = m.pct_change().fillna(0)
    fora = caixa.reindex(ret.index).fillna(0) if caixa is not None else 0 * ret
    ret, sinal, fora = ret.loc["2000":], sinal.loc["2000":], fora.loc["2000":]
    curvas = {"estratégia": (1 + ret.where(sinal, fora)).cumprod(), "comprar e segurar": (1 + ret).cumprod()}
    if caixa is not None:
        curvas["só CDI"] = (1 + fora).cumprod()
    anos = len(ret) / 12
    FABER[nome] = curvas
    print(f"   {nome}: comprado em {pc(sinal.mean())} dos meses")
    for rot, s_ in curvas.items():
        print(f"   {nome:<9} {rot:<18} {ps(s_.iloc[-1] ** (1 / anos) - 1, 1)} ao ano | maior queda {pc((s_ / s_.cummax() - 1).min())}")

## Parte 8: Setup MK no gráfico de 60 minutos

In [ ]:
r60, ref60 = [], []
for t in INTRA:
    x = limpa(H60[t])
    cc = x["Close"]
    res = setup_mk(x["Open"].to_numpy(), x["High"].to_numpy(), x["Low"].to_numpy(), cc.to_numpy(),
                   cc.ewm(span=9, adjust=False).mean().to_numpy(), cc.ewm(span=20, adjust=False).mean().to_numpy())
    r60 += [r for r, _, _ in res]
    ref60 += [cc.pct_change().mean() * dd for _, dd, _ in res]
r60 = np.array(r60)
print(f"\nParte 8. Setup MK no 60 minutos (8 ativos, ~2 anos): {nn(len(r60))} operações | com custo {ps(r60.mean() - CUSTO)}"
      f" | acerto {pc((r60 > 0).mean())} | segurar pelo mesmo número de barras {ps(np.mean(ref60))}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a Parte 5, que classifica as fases do diamante de Chuck Dukas com `np.select` e as médias de 50 e 200. Mostre qual condição corresponde a cada uma das seis fases e por que `c.shift(-20)`, que olha 20 pregões à frente, não é um erro de olhar o futuro nesta Parte." Confira com a Parte 5, que imprime a fase altista com 42% dos dias e média de +1,02% nos 20 pregões seguintes, e a baixista com +1,99%.
2. **Adaptar:** "Numa célula nova, abaixo da Parte 3, baixe [ATIVO] com o ticker do Yahoo Finance, de 2005 até `FIM`, passe pela função `limpa` e rode `opera` só nele. Mostre a linha do IFR(2) de Connors no mesmo formato da Parte 3. Não mude as listas `B3` e `EUA`, o `CUSTO` nem as regras de `opera`. Qualquer conclusão sobre [ATIVO] precisa do teste fora da amostra (Capítulo 27)." Antes de olhar o ativo novo, confira se a Parte 3 continua mostrando 11.064 operações e +0,39% com custo no IFR(2) de Connors e 3.991 operações no %b de Bollinger < 0 (bandas com o desvio da população, `ddof=0`), como na Tabela 25.1.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. No começo do notebook, troque o custo (CUSTO) para 0,20% ida e volta. Quais estratégias continuam acima de zero?
2. No IFR(2), use o nível 5 em vez de 10. O acerto sobe? E o número de operações?
3. Na Parte 7, troque a média de 10 meses por 12 meses. O resultado do Faber muda muito? Isso é sinal de regra estável ou de regra frágil?
4. Escreva o seu setup favorito no formato da função opera() e compare com a Tabela 25.1.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.